# Mapping Exposure - Infraestructure
This notebook explores different sources to obtain georeferenced data about infrastructure

## Open Street Maps - OSM

OpenStreetMap is a free, editable map of the whole world that is being built by volunteers largely from scratch and released with an open-content license.
From this source one can download Points of Interest (POIs) like schools, hospitals, churchs, etc. Also, the entire road network of a city can be downloaded. 
[Source](https://wiki.openstreetmap.org/wiki/About_OpenStreetMap#:~:text=OpenStreetMap%20is%20a%20free%2C%20editable,of%20our%20underlying%20map%20data.)

OSM has an API for accessing to their data. However, this course uses [OSMnx](https://geoffboeing.com/publications/osmnx-paper/), a Python package that simplifies the querying process. 



### Downloading POIs
OSMnx has the feature module that downloads OpenStreetMap geospatial features’ geometries and attributes.
>Retrieve points of interest, building footprints, transit lines/stops, or any other map features from OSM, including their geometries and attribute data, then construct a GeoDataFrame of them. You can use this module to query for nodes, ways, and relations (the latter of type “multipolygon” or “boundary” only) by passing a dictionary of desired OSM tags.

This example will continue working with 02/06/2023 earthquake in Türkiye. The information will be downloaded for a square of 50km around the epicenter.

In [ ]:
import osmnx as ox
import geopandas as gpd
from shapely import Point, Polygon
import folium

Considering the epicenter as: 38.011°N 37.196°E:
1. Create a GeoDataFrame for the epicenter
2. Project the point into a projected CRS
3. Generate a buffer of 50km around the epicenter
4. Generate the bounding box to extract the information

In [ ]:
d = {'name': ['epicenter'], 'geometry': [Point(37.196, 38.011)]}
gdf = gpd.GeoDataFrame(d, crs="EPSG:4326")
gdf = gdf.to_crs('EPSG:23036')
gdf['buffer'] = gdf.geometry.apply(lambda x: x.buffer(50000))
gdf = gdf.set_geometry('buffer')
gdf = gdf.to_crs('EPSG:4326')

In [ ]:
bounds = gdf.bounds.loc[0]
north, south, east, west = bounds.maxy, bounds.miny, bounds.minx, bounds.maxx
gdf.loc[1, 'name'] = 'epicenter_rectangle'
gdf.loc[1, 'buffer'] = Polygon(((east,north), (west, north), (west, south), (east, south)))

In [ ]:
m = gdf.explore()
m

Acoording to the [documentation](https://osmnx.readthedocs.io/en/stable/user-reference.html#osmnx.features.features_from_bbox), the tags should be specified as follows:
>**tags** (dict) – Dict of tags used for finding elements in the selected area. Results returned are the union, not intersection of each individual tag. Each result matches at least one given tag. The dict keys should be OSM tags, (e.g., building, landuse, highway, etc) and the dict values should be either True to retrieve all items with the given tag, or a string to get a single tag-value combination, or a list of strings to get multiple values for the given tag. For example, tags = {‘building’: True} would return all building footprints in the area. tags = {‘amenity’:True, ‘landuse’:[‘retail’,’commercial’], ‘highway’:’bus_stop’} would return all amenities, landuse=retail, landuse=commercial, and highway=bus_stop.

In order to download all POIS specified as hospitals, one would do the following:

In [ ]:
tags = {'amenity': 'hospital'}

In [ ]:
hospitals = ox.features_from_polygon(gdf.geometry.loc[1], tags=tags)
hospitals #the result is a GeoDataFrame

As it is shown, the element type, in this particular case, can be a `way` or a `node` in this case, more information about ways can be found [here](https://wiki.openstreetmap.org/wiki/Way).
The hospitals can be represented as a point or a polygon. 

As it can be seen in the map below, "Afşin Devlet Hastanesi" hospital is being double counted. This is expected due to the nature of the data, crowd sourced. Depending on the usage of the data, one might need to clean the dataset with more or less effort. For example, if the goal is to register all the hospitals that might had been affected, then, double-counting might not be an issue. However, if the goal is to estimate number of affected hospitals, double-counting will need to be eliminated. 

In [ ]:
hospitals.explore()

In [ ]:
# If the goal is to download all the footprints of buildings, one can avoid specifying a tag and use True
tags = {'building': True}
footprints = ox.features_from_polygon(gdf.geometry.loc[1], tags=tags)

In [ ]:
footprints.head()

In [ ]:
print('With OSM {} buildings were downloaded'.format(len(footprints)))

In [ ]:
footprints.amenity.value_counts()

In [ ]:
footprints.explore()

Let's explore the limimations of this data with an example. When downloading all the amenities tagged as `restaurant`, only 17 places are found. It seems unlikely that there are 17 restaurants in the entire region. So, it is worth querying all the amenities. Unfortunately, the result is the same and the quality of the data for restaurants is not good. 

In [ ]:
tags = {'amenity': 'restaurant'}
restaurant = ox.features_from_polygon(gdf.geometry.loc[1], tags=tags)

In [ ]:
len(restaurant)

In [ ]:
tags = {'amenity': True}
amenities = ox.features_from_polygon(gdf.geometry.loc[1], tags=tags)

In [ ]:
amenities['amenity'].value_counts()

### Debate
- What do you think about this dataset?
- In which crisis related task will it become handy?
- Do you see any other limitation? 
- Which are your proposals to clean this type of data? 

#### Note
OSM data can be downloaded in other ways as well, for example using Qgis or even ChatGPT. This course decided to use OSMnx as it more flexible. 

## Overture Maps Foundation
According to they webpage overture creates reliable, easy-to-use, and interoperable open map data.
They build their maps collaboratively incorporating map data from multiple sources including Overture Members, civic organizations, and open data sources.

Overture deals with the problem that multiple dataset reference the same real-world entities using their own conventions and vocabulary, making them difficult to merge and combine. They simplify interoperability by providing a system that links entities from different data sets to the same real-world entities.

Overture is currently focused on building the following vecore data layers:
- administrative boundaries
- base: water, land, land use, infrastructure, land cover
- buildings
- places
- transportation

More information about their data can be found in their [documentation](https://docs.overturemaps.org/).

### Buildings layer

The Overture Maps buildings theme describes human-made structures with roofs or interior spaces that are permanently or semi-permanently in one place (source: OSM building definition). The theme includes two feature types:

- **building**: The most basic form of a building feature. The geometry is expected to be the most outer footprint -- or roofprint if traced from satellite/aerial imagery -- of a building.
- **building_part**: A single part of a building. Building parts may have the same properties as buildings. A building part is associated with a parent building via a building_id.

The documentation offers a detailed explanation about the data schema for each of the layers. For the builing layer, it can be read [here](https://docs.overturemaps.org/schema/reference/buildings/building/).

The OMF’s Buildings layer includes over 780M unique buildings footprints worldwide. This layer has been developed by combining various open data projects including OpenStreetMap, Microsoft AI-Generated building footprints, and Esri.[source](https://overturemaps.org/overture-maps-foundation-releases-first-world-wide-open-map-dataset/)

The following example downloads building data from Overture using their python package which is a Beta version.

In [ ]:
import pandas as pd
import geopandas as gpd
import overturemaps 
from shapely import wkb
from lonboard import Map, PolygonLayer
from lonboard.colormap import apply_categorical_cmap
pd.set_option('display.max_columns', 500) #allows to inspect all the columns present in the dataframe

import matplotlib.pyplot as plt

In [ ]:
# specify bounding box
bbox = east, south, west, north #use the 50km rectangle

In [ ]:
#Download builidngs from overture
table = overturemaps.record_batch_reader("building", bbox).read_all()
table = table.combine_chunks()
# convert to dataframe
df = table.to_pandas()

In [ ]:
# dataframe to geodataframe, set crs
gdf_overture = gpd.GeoDataFrame(
    df, 
    geometry=df['geometry'].apply(wkb.loads), 
    crs="EPSG:4326"
)

In [ ]:
gdf_overture.head()

Overture uses different sources to create their maps data. Below, it can be seen that for this region, footprints come from OpenStreet Map and from Microsoft ML Buildings. 

Below, there is a map that colors the footprint based on its source.

In [ ]:
# extract the data point primary source to compare with OSM
gdf_overture['dataset'] = gdf_overture.sources.apply(lambda x: x[0]['dataset'])

In [ ]:
gdf_overture.dataset.value_counts()

In [ ]:
gdf_overture[['geometry', 'dataset']].iloc[0:100000].explore(column = 'dataset')

### Places
The Overture Maps places theme contains datasets with point representations of real-world facilities, services, businesses or amenities.

OMF’s Places data layer includes over 59 million POI records that have not previously been released as open data. This dataset, derived from data contributed to OMF by founding members **Meta** and **Microsoft**, provides a significant baseline of worldwide places data. The Overture community will combine the best data from all available resources, including open government data, crowdsourced local mapping data, AI/ML techniques and more to improve, update and extend the data on an ongoing basis.  The Places dataset is licensed under CDLA Permissive v2.0, a permissive open data license, and can be freely used by any map builders or location service providers.

The following example shows how to download the place layer for the area of interest. 

In [ ]:
#Download builidngs from overture
table_places = overturemaps.record_batch_reader("place", bbox).read_all()
table_places = table_places.combine_chunks()
# convert to dataframe
df = table_places.to_pandas()

In [ ]:
# dataframe to geodataframe, set crs
gdf_overture_plc = gpd.GeoDataFrame(
    df, 
    geometry=df['geometry'].apply(wkb.loads), 
    crs="EPSG:4326"
)

In [ ]:
# extract the data point primary source to compare with OSM
gdf_overture_plc['dataset'] = gdf_overture_plc.sources.apply(lambda x: x[0]['dataset'])

#extract the category
gdf_overture_plc.loc[gdf_overture_plc['categories'].notnull(), 'category'] = gdf_overture_plc.loc[gdf_overture_plc['categories'].notnull(), 'categories'].apply(lambda x: x['main'])

#extract the primary name
gdf_overture_plc['primary_name'] = gdf_overture_plc.names.apply(lambda x: x['primary'])

In [ ]:
gdf_overture_plc.head()

The code below explores:
- Where the data comes from. In this case it is from Meta and Microsoft. 
- The categories, a list with all OMF categories can be found [here](https://github.com/OvertureMaps/schema/blob/main/task-force-docs/places/overture_categories.csv). This example is interested in extracting health care related facilities.
- The confidence of the existence of the place. Confidence is a number between 0 and 1. 0 means that OMF is sure that the place doesn't exist (anymore). 1 means that OMF is sure that the place exists. If there's no value for the confidence, it means that OMF don't have any confidence information.

In [ ]:
gdf_overture_plc.dataset.value_counts()

In [ ]:
for cat in gdf_overture_plc.category.unique():
    if cat == cat:
        if ('hospital' in cat) or ('health' in cat):
            print(cat)

In [ ]:
#create a map to show health related facilities
m = gdf_overture_plc[gdf_overture_plc['category'].isin(['health_and_medical', 'hospital', 'counseling_and_mental_health'])][['category', 'dataset', 'geometry', 'primary_name', 'confidence', 'id']].explore(column = 'category', name = 'overture')
hospitals.explore(m = m, color = 'red', name = 'osm')
folium.LayerControl().add_to(m)
m

In [ ]:
#create a map to show health related facilities that have a confidence greater than 0.5
m = gdf_overture_plc[(gdf_overture_plc['category'].isin(['health_and_medical', 'hospital', 'counseling_and_mental_health'])) &
                     (gdf_overture_plc['confidence']>0.5)][['category', 'dataset', 'geometry', 'primary_name', 'confidence', 'id']].explore(column = 'category', name = 'overture')
hospitals.explore(m = m, color = 'red', name = 'osm')
folium.LayerControl().add_to(m)
m